# Numerical Differentiation and Richardson Extrapolation

## Part 1: Forward, Backward and Central Differentiation

$$\text{forward:}\quad f'(x) \simeq \frac{f(x+h)-f(x)}{h} \tag{4.6}$$
$$\text{backward:}\quad f'(x) \simeq \frac{f(x)-f(x-h)}{h} \tag{4.7}$$
$$\text{central:}\quad f'(x) \simeq \frac{f(x+h)-f(x-h)}{2h} \tag{4.8}$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.polynomial import Polynomial

### Task 1: difference formulas and error function

In [ ]:
def forward_diff(f, h, x):
    return (f(x + h) - f(x)) / h

def backward_diff(f, h, x):
    return (f(x) - f(x - h)) / h

def central_diff(f, h, x):
    return (f(x + h) - f(x - h)) / (2 * h)

In [ ]:
def error_1(f, f_prime, h, x):
    """Error of the forward, backward and central methods at point x for each step size in h."""
    y_correct = f_prime(x)
    f_error = np.array([forward_diff(f, h_i, x) - y_correct for h_i in h])
    b_error = np.array([backward_diff(f, h_i, x) - y_correct for h_i in h])
    c_error = np.array([central_diff(f, h_i, x) - y_correct for h_i in h])
    print(pd.DataFrame({"h": h, "Forward": f_error, "Backward": b_error, "Central": c_error}))
    return f_error, b_error, c_error

### Plotting: the polynomial and its actual derivative

In [ ]:
p = Polynomial([3.5, 2.0, -10.0, -6.0, 6.0, 3.0])
p_prime = p.deriv(1)

fig, ax = plt.subplots()
ax.axhline(y=0, color='k')
ax.plot(*p.linspace(domain=[-2.5, 2.0]), label='Function')
ax.plot(*p_prime.linspace(domain=[-2.5, 2.0]), label='Derivative')
ax.legend()
plt.show()

### Comparing the three methods for different step sizes

In [ ]:
def compare_methods(f, f_prime, h, x_range=(-2.2, 2.3)):
    x = np.linspace(*x_range, 50, endpoint=True)
    fig, ax = plt.subplots()
    ax.axhline(y=0, color='k')
    ax.plot(x, forward_diff(f, h, x), label=f'Forward; h={h}')
    ax.plot(x, backward_diff(f, h, x), label=f'Backward; h={h}')
    ax.plot(x, central_diff(f, h, x), label=f'Central; h={h}')
    ax.plot(*f_prime.linspace(domain=list(x_range)), label='Actual')
    ax.set_title(f'Numerical derivatives, h = {h}')
    ax.legend()
    plt.show()

for h in [0.5, 0.09, 0.01]:
    compare_methods(p, p_prime, h)

### Error vs. step size at x = 2.0

In [ ]:
h = np.array([3.5, 0.55, 0.3, .17, 0.1, 0.055, 0.03, 0.017, 0.01])
err = error_1(p, p_prime, h, 2.0)

fig, ax = plt.subplots()
ax.axhline(y=0, color='k')
ax.plot(h, err[0], label='Forward')
ax.plot(h, err[1], label='Backward')
ax.plot(h, err[2], label='Central')
ax.set_xlabel('h'); ax.set_ylabel('error')
ax.legend()
plt.show()

## Part 2: Richardson Extrapolation

Let
$$D_h = \frac{f(x_1+h)-f(x_1-h)}{2h} \tag{5.1}$$

Expanding with Taylor's theorem and subtracting gives
$$D_h = f'(x_1) + \frac{f'''(x_1)}{6}h^2 + \frac{f^{(5)}(x_1)}{120}h^4 + O(h^6) \tag{5.5}$$

Using step sizes $h$ and $h/2$ and eliminating the $h^2$ term:
$$D_h^{(1)} = \frac{4D_{h/2} - D_h}{3} = f'(x_1) - \frac{f^{(5)}(x_1)}{480}h^4 + O(h^6) \tag{5.8}$$

So the error drops from order 2 to order 4.

### Task 2: `dh`, `dh1` and `error_2`

In [ ]:
def dh(f, h, x):
    """Central difference derivative of f at x (eq. 5.1)."""
    return (f(x + h) - f(x - h)) / (2 * h)

def dh1(f, h, x):
    """Richardson extrapolated derivative of f at x (eq. 5.8)."""
    return (4 * dh(f, h / 2, x) - dh(f, h, x)) / 3

In [ ]:
def error_2(f, hs, x_i):
    """Errors of dh and dh1 at x_i for each step size in hs."""
    y_actual = f.deriv(1)(x_i)
    diff_error = [dh(f, h, x_i) - y_actual for h in hs]
    diff2_error = [dh1(f, h, x_i) - y_actual for h in hs]
    print(pd.DataFrame({"h": hs, "Diff": diff_error, "Diff2": diff2_error}))
    return diff_error, diff2_error

### Tests

In [ ]:
def draw_graph(f, ax, domain=[-10, 10], label=None):
    ax.plot(*f.linspace(domain=domain), label=label)

p = Polynomial([3.0, 4.0, -14.0, -10.0, 3.0, 1.5])
p_prime = p.deriv(1)

fig, ax = plt.subplots()
ax.axhline(y=0, color='k')
draw_graph(p, ax, [-2.5, 2.0], 'Function')
draw_graph(p_prime, ax, [-2.5, 2.0], 'Derivative')
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.axhline(y=0, color='k')
draw_graph(p_prime, ax, [-2.5, 2.0], 'Actual')

x = np.linspace(-2.5, 2.0, 50, endpoint=True)
for h in [1.5, 0.1]:
    ax.plot(x, dh1(p, h, x), label=f'Richardson; h={h}')
ax.legend()
plt.show()

In [ ]:
hs = np.array([3.5, 0.55, 0.3, .17, 0.1, 0.055, 0.03, 0.017, 0.01])
e1, e2 = error_2(p, hs, 2.0)

fig, ax = plt.subplots()
ax.axhline(y=0, color='k')
ax.plot(hs, e1, label='dh (central)')
ax.plot(hs, e2, label='dh1 (Richardson)')
ax.set_xlabel('h'); ax.set_ylabel('error')
ax.legend()
plt.show()

## Daily Evaluation

In [ ]:
p = Polynomial([2.0, 1.0, -6.0, -2.0, 2.5, 1.0])
p_prime = p.deriv(1)
print(p)

In [ ]:
# (a) the function and its derivative
fig, ax = plt.subplots()
ax.axhline(y=0, color='k')
draw_graph(p, ax, [-2.5, 2.0], 'Function')
draw_graph(p_prime, ax, [-2.5, 2.0], 'Derivative')
ax.legend()
plt.show()

In [ ]:
# forward, backward and central differences with h = 0.1
compare_methods(p, p_prime, 0.1, x_range=(-2.0, 2.3))

In [ ]:
# (e) error vs. h at x = 2.0
h = np.array([0.55, 0.3, .17, 0.1, 0.055, 0.03, 0.017, 0.01])
err = error_1(p, p_prime, h, 2.0)

fig, ax = plt.subplots()
ax.axhline(y=0, color='k')
ax.plot(h, err[0], label='Forward')
ax.plot(h, err[1], label='Backward')
ax.plot(h, err[2], label='Central')
ax.set_xlabel('h'); ax.set_ylabel('error')
ax.legend()
plt.show()